In [4]:
# ============================================
# MLflow Starter - Autolog, Manual Log, Store & Load
# Dataset: Breast Cancer (sklearn built-in)
# ============================================

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

# --- Load Dataset ---
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target
print(f"Dataset shape: {df.shape}")
print(f"Target classes: {dict(enumerate(data.target_names))}")
print(df.head())

X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# --- PART 1: Autolog ---
print("=" * 50)
print("PART 1: MLflow Autolog")
print("=" * 50)

mlflow.autolog()

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
predictions = rf.predict(X_test)
print(f"Autolog Accuracy: {np.mean(predictions == y_test):.4f}")

# --- PART 2: Manual Logging ---
print("\n" + "=" * 50)
print("PART 2: Manual Logging")
print("=" * 50)

mlflow.autolog(disable=True)

with mlflow.start_run(run_name="manual_rf_breast_cancer") as run:
    rf2 = RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42)
    rf2.fit(X_train, y_train)
    preds = rf2.predict(X_test)

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    # Log parameters manually
    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 8)

    # Log metrics manually
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", prec)
    mlflow.log_metric("recall", rec)
    mlflow.log_metric("f1_score", f1)

    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # Log model with signature
    signature = infer_signature(X_train, rf2.predict(X_train))
    mlflow.sklearn.log_model(rf2, "model", signature=signature)

    print(f"\nRun ID: {run.info.run_id}")

# --- PART 3: Load the saved model ---
print("\n" + "=" * 50)
print("PART 3: Load Saved Model")
print("=" * 50)

model_uri = f"runs:/{run.info.run_id}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_preds = loaded_model.predict(X_test)
print(f"Loaded Model Accuracy: {accuracy_score(y_test, loaded_preds):.4f}")
print("Model loaded and verified successfully!")

Dataset shape: (569, 31)
Target classes: {0: np.str_('malignant'), 1: np.str_('benign')}
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   

2026/04/01 00:19:44 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/04/01 00:19:44 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8ca0e7c15f8e40b486f1f1362174ed5e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/04/01 00:19:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Autolog Accuracy: 0.9650

PART 2: Manual Logging


2026/04/01 00:19:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 00:19:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Accuracy: 0.9650
Precision: 0.9667
Recall: 0.9775
F1 Score: 0.9721

Run ID: 1b3309fb29604fa184da13f5b47287fe

PART 3: Load Saved Model


Loaded Model Accuracy: 0.9650
Model loaded and verified successfully!
